In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/telco_churn.csv", delimiter="|")

df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
df['churn'] = df['Churn Value']
df = df.drop(columns=['Churn Label'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x='churn', data=df)
plt.title("Churn Distribution")
plt.show()

In [ ]:
df.columns = df.columns.str.lower().str.replace(" ", "_")
df.head()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')
df['total_charges'].fillna(df['total_charges'].median(), inplace=True)

In [ ]:
df = df.drop(columns=[
    'customerid', 
    'lat_long',
    'latitude',
    'longitude',
    'churn_reason',
    'churn_label',
    'count'
], errors='ignore')

In [ ]:
df = df.drop(columns=['churn_score'], errors='ignore')

In [ ]:
y = df['churn_value']
X = df.drop(columns=['churn_value'])

In [ ]:
X = pd.get_dummies(X, drop_first=True)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# y_pred = model.predict(X_test)


# print(confusion_matrix(y_test, y_pred))
# print(classification_report(y_test, y_pred))
y_prob = model.predict_proba(X_test)[:, 1]
y_pred_adjusted = (y_prob > 0.35).astype(int)

print(confusion_matrix(y_test, y_pred_adjusted))
print(classification_report(y_test, y_pred_adjusted))

In [ ]:
importance = pd.Series(
    model.coef_[0], 
    index=X.columns
).sort_values(ascending=False)

importance.head(10)

In [ ]:
importance.tail(10)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Calculate ROC values
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# Calculate AUC
auc_score = roc_auc_score(y_test, y_prob)

print("AUC Score:", auc_score)

# Plot ROC Curve
plt.figure()
plt.plot(fpr, tpr, label="Logistic Regression (AUC = %0.3f)" % auc_score)
plt.plot([0, 1], [0, 1])  # Diagonal reference line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Churn Prediction")
plt.legend()
plt.show()

plt.savefig("../images/roc_curve.png", bbox_inches='tight')

In [ ]:
# Create importance dataframe
importance_df = pd.Series(model.coef_[0], index=X.columns)
importance_df = importance_df.sort_values()

# Plot top 10 positive & negative features
plt.figure()
importance_df.tail(10).plot(kind='barh')
plt.title("Top 10 Features Increasing Churn Risk")
plt.xlabel("Coefficient Value")
plt.show()

plt.figure()
importance_df.head(10).plot(kind='barh')
plt.title("Top 10 Features Reducing Churn Risk")
plt.xlabel("Coefficient Value")
plt.show()

plt.savefig("../images/feature_importance_positive.png", bbox_inches='tight')
plt.savefig("../images/feature_importance_negative.png", bbox_inches='tight')